### PandasDataFrameOutputParser pg.185

In [2]:
#!pip --version

In [3]:
from dotenv import load_dotenv

load_dotenv()

True

In [4]:
import pandas as pd
from langchain_openai import ChatOpenAI
from typing import Any,Dict
import pprint

from langchain_classic.output_parsers import PandasDataFrameOutputParser
from langchain_core.prompts import PromptTemplate

In [5]:
model = ChatOpenAI(
    temperature=0,
    model= "gpt-3.5-turbo"
)

In [6]:

# 출력을 Dict로 변환 및 출력 형식 지정
def format_parser_output(parser_output: Dict[str, Any]) -> None:

    for key in parser_output.keys():
        parser_output[key] = parser_output[key].to_dict()
    return pprint.PrettyPrinter(width=4,  compact=True).pprint(parser_output)

In [10]:

# 타이타닉 자료를 데이터프레임으로 가져오기
df = pd.read_csv("./data/titanic.csv")
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [11]:

parser = PandasDataFrameOutputParser(dataframe=df)
print(parser.get_format_instructions())

The output should be formatted as a string as the operation, followed by a colon, followed by the column or row to be queried on, followed by optional array parameters.
1. The column names are limited to the possible columns below.
2. Arrays must either be a comma-separated list of numbers formatted as [1,3,5], or it must be in range of numbers formatted as [0..4].
3. Remember that arrays are optional and not necessarily required.
4. If the column is not in the possible columns or the operation is not a valid Pandas DataFrame operation, return why it is invalid as a sentence starting with either "Invalid column" or "Invalid operation".

As an example, for the formats:
1. String "column:num_legs" is a well-formatted instance which gets the column num_legs, where num_legs is a possible column.
2. String "row:1" is a well-formatted instance which gets row 1.
3. String "column:num_legs[1,2]" is a well-formatted instance which gets the column num_legs for rows 1 and 2, where num_legs is a p

In [14]:
#열 작업 예시

df_query = "Age column을 조회해 주세요"

# 프롬프트 템플릿 작성
Prompt = PromptTemplate(
    template= "Answer the user query.\n{format_instructions}\n{question}\n",
    input_variables= ["question"],
    partial_variables= {
        "format_instructions": parser.get_format_instructions()

    },
)

chain = Prompt | model | parser

parser_output = chain.invoke({"question": df_query})

format_parser_output(parser_output)

{'Age': {0: 22.0,
         1: 38.0,
         2: 26.0,
         3: 35.0,
         4: 35.0,
         5: nan,
         6: 54.0,
         7: 2.0,
         8: 27.0,
         9: 14.0,
         10: 4.0,
         11: 58.0,
         12: 20.0,
         13: 39.0,
         14: 14.0,
         15: 55.0,
         16: 2.0,
         17: nan,
         18: 31.0,
         19: nan,
         20: 35.0,
         21: 34.0,
         22: 15.0,
         23: 28.0,
         24: 8.0,
         25: 38.0,
         26: nan,
         27: 19.0,
         28: nan,
         29: nan,
         30: 40.0,
         31: nan,
         32: nan,
         33: 66.0,
         34: 28.0,
         35: 42.0,
         36: nan,
         37: 21.0,
         38: 18.0,
         39: 14.0,
         40: 40.0,
         41: 27.0,
         42: nan,
         43: 3.0,
         44: 19.0,
         45: nan,
         46: nan,
         47: nan,
         48: nan,
         49: 18.0,
         50: 7.0,
         51: 21.0,
         52: 49.0,
         53: 29.0,
    

In [15]:
# 행 조회 예시
df_query = "Retrieve the first row."

# 체인 실행
parser_output = chain.invoke({"question": df_query})

# 결과 출력
format_parser_output(parser_output)

{'0': {'Age': 22.0,
       'Cabin': nan,
       'Embarked': 'S',
       'Fare': 7.25,
       'Name': 'Braund, '
               'Mr. '
               'Owen '
               'Harris',
       'Parch': 0,
       'PassengerId': 1,
       'Pclass': 3,
       'Sex': 'male',
       'SibSp': 1,
       'Survived': 0,
       'Ticket': 'A/5 '
                 '21171'}}
